In [15]:
#Part A — Profile and Audit the Dataset

In [7]:
!pip install ydata-profiling

In [8]:
#1.upload csv
from google.colab import files
uploaded=files.upload()

Saving healthtrack_raw.csv to healthtrack_raw (1).csv


In [12]:
#2.load and generate report
import pandas as pd
from ydata_profiling import ProfileReport

df=pd.read_csv("healthtrack_raw.csv")
profile=ProfileReport(df,title="healthtrack_profile")
profile.to_file("healthtrack_profile.html")
print("Done!")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 12/12 [00:00<00:00, 41.85it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Done!


In [14]:
from google.colab import files
files.download("healthtrack_profile.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
#3.Answer
total = len(df)

# 1. Missing patient_id
missing_pid = df["patient_id"].isna().sum()
pct_pid = round((missing_pid / total) * 100, 2)

# 2. Missing readmission_30d
missing_label = df["readmission_30d"].isna().sum()
pct_label = round((missing_label / total) * 100, 2)

# 3. Sentinel value
sentinel_bill = (df["total_bill_usd"] == -9999).sum()

# 4. Department with highest readmission rate
dept_readmission = df.groupby("department")["readmission_30d"].mean().sort_values(ascending=False)

print(f"Total rows: {total}")
print(f"Missing patient_id: {missing_pid} ({pct_pid}%)")
print(f"Missing readmission_30d: {missing_label} ({pct_label}%)")
print(f"Sentinel bills (-9999): {sentinel_bill}")
print(f"\nReadmission rate by department:\n{dept_readmission}")

Total rows: 300
Missing patient_id: 25 (8.33%)
Missing readmission_30d: 30 (10.0%)
Sentinel bills (-9999): 15

Readmission rate by department:
department
Oncology            0.510204
General Medicine    0.444444
Orthopedics         0.433962
Neurology           0.355932
Cardiology          0.290909
Name: readmission_30d, dtype: float64


## Audit Findings

- **Missing patient_id:** 25 rows (8.33% of dataset)
- **Missing readmission_30d:** 30 rows (10%) — blocks model training as it is the target label
- **Sentinel bills (-9999):** 15 rows — EHR error code, not a real billing value
- **Highest readmission department:** Oncology at 51%

**CMO Note:** 30 patient records are missing their readmission outcome, which prevents
the AI model from learning from 10% of this year's discharges and risks missing early
intervention opportunities for high-risk patients.